# Parameter Derivation

### Stress Profiling
This corresponds to the first part of the parameter derivation procedure. We have used a video decoder with each frame being looped 5 times in the decorder (--work=5).
The workload has been run 100.000 times without any stress on the shared components, then with stress on the cache and then on the memory. Finally, an exeution wihtout any stress after the previous tests. These has been executed for single and multi-core.

#### Metrics
- **Execution time** = `end_ns - start_ns` (wall time of the job, upper bound on its CPU time)
- **Dispatch latency** = `start_ns - release_ns`

All executed jobs are included, outliers too. Skipped activations have no start/end timestamps, so there is nothing to measure for them.

In [ ]:
from pathlib import Path
import pandas as pd

RESULTS = Path("../results")
SCENARIOS = {"single_core": ["instance0"], "multi_core": ["instance0", "instance1"]}
CONDITIONS = ["baseline1", "cache", "memory", "baseline2"]
STRESS = ["cache", "memory"]
# statistics shown per metric (quantile levels; 1.0 = max)
METRICS = {
    "exec_ms":     ("Execution time (ms)",   {"p50": .50, "p99": .99, "p99.9": .999, "max": 1.0}),
    "dispatch_us": ("Dispatch latency (µs)", {"p50": .50, "max": 1.0}),
}

frames = []
for scenario, instances in SCENARIOS.items():
    for condition in CONDITIONS:
        for inst in instances:
            df = pd.read_csv(RESULTS / scenario / condition / f"{inst}.csv")
            df["scenario"], df["condition"] = scenario, condition
            frames.append(df)
data = pd.concat(frames, ignore_index=True)

print("skipped activations (no timestamps):")
print(data.groupby(["scenario", "condition"], sort=False).skipped.sum().to_string())

data = data[data.skipped == 0].copy()
data["exec_ms"] = (data.end_ns - data.start_ns) / 1e6
data["dispatch_us"] = (data.start_ns - data.release_ns) / 1e3

In [ ]:
def stats_table(scenario):
    d = data[data.scenario == scenario]
    tbl = pd.concat(
        {name: d.groupby("condition")[m].quantile(list(stats.values())).unstack()
                .set_axis(list(stats), axis=1)
         for m, (name, stats) in METRICS.items()}, axis=1)
    # spread: how far the worst job is from the typical one
    for name, _ in METRICS.values():
        tbl[(name, "max − p50")] = tbl[(name, "max")] - tbl[(name, "p50")]
        tbl[(name, "max vs p50 %")] = (tbl[(name, "max")] / tbl[(name, "p50")] - 1) * 100
    order = [c for name, _ in METRICS.values() for c in tbl.columns if c[0] == name]
    return tbl.loc[CONDITIONS, order]

def pct_table(tbl):
    tbl = tbl.drop(columns=[c for c in tbl.columns if c[1] == "max vs p50 %"])
    rows = {}
    for ref, name in [("baseline1", "before"), ("baseline2", "after")]:
        for s in STRESS:
            rows[f"{s} vs baseline {name}"] = (tbl.loc[s] / tbl.loc[ref] - 1) * 100
    return pd.DataFrame(rows).T

tables, pcts = {}, {}
for scenario in SCENARIOS:
    tables[scenario] = stats_table(scenario)
    pcts[scenario] = pct_table(tables[scenario])

### Single-core

In [ ]:
display(tables["single_core"].style.format("{:,.3f}"))
display(pcts["single_core"].style.format("{:+.2f}%"))

### Multi-core (instance0 and instance1 pooled)

In [ ]:
display(tables["multi_core"].style.format("{:,.3f}"))
display(pcts["multi_core"].style.format("{:+.2f}%"))

### How often are large dispatch latencies (W)?
W = `start_ns - release_ns`. Computed per instance, because the miss rate is defined per task.
If jobs with a large W are only a handful per 100,000, they form a rare floor of misses. If they are more frequent, they could dominate the miss rate at the tolerance levels 10⁻² and 10⁻³.
The `share of 1e-x` columns give the W > 5 ms rate as a fraction of each tolerance (e.g. 0.024 = 2.4% of the allowed misses).

In [ ]:
THRESHOLDS_MS = [5, 15]
TOLERANCES = [1e-2, 1e-3]

def w_tail(g):
    w = g.dispatch_us
    row = {"jobs": len(w), "W p99 (µs)": w.quantile(.99), "W p99.9 (µs)": w.quantile(.999)}
    for t in THRESHOLDS_MS:
        row[f"W > {t} ms"] = int((w > t * 1_000).sum())
    for t in THRESHOLDS_MS:
        row[f"rate W > {t} ms"] = row[f"W > {t} ms"] / len(w)
    for tol in TOLERANCES:
        row[f"share of {tol:.0e}"] = row[f"rate W > {THRESHOLDS_MS[0]} ms"] / tol
    return pd.Series(row)

w_tbl = (data.groupby(["scenario", "instance_id", "condition"], sort=False)[["dispatch_us"]]
             .apply(w_tail).astype({f"W > {t} ms": int for t in THRESHOLDS_MS} | {"jobs": int}))
w_tbl.style.format({"W p99 (µs)": "{:,.1f}", "W p99.9 (µs)": "{:,.1f}",
                    **{f"rate W > {t} ms": "{:.1e}" for t in THRESHOLDS_MS},
                    **{f"share of {tol:.0e}": "{:.3f}" for tol in TOLERANCES}})